# Qwen3.8-27B on Kaggle with Prebuilt `llama.cpp` (`llama-server`) + OpenAI Endpoint

Uses official **prebuilt `llama.cpp` binaries** (no compilation!) to run **`Qwen3.8-27B`** and expose a native OpenAI-compatible API (`/v1/chat/completions`, `/v1/completions`, `/v1/models`, streaming SSE) via Cloudflare Tunnel.

### Recommended Kaggle Settings
- **Accelerator:** **GPU T4 x2** (32 GB VRAM total — ideal for `UD-IQ4_XS` or `UD-Q4_K_M`) or **GPU P100** (16 GB VRAM — use `UD-IQ3_S` or `UD-IQ4_XS` with smaller context).
- **Internet:** **ON**

## 1. Configuration & Hardware / Disk Check

In [ ]:
import os, shutil, subprocess

# Recommended quants for unsloth/Qwen3.8-27B-GGUF:
# - T4 x2 (32 GB VRAM): "Qwen3.8-27B-UD-IQ4_XS.gguf" (~14.3 GB) or "Qwen3.8-27B-UD-Q4_K_M.gguf" (~17 GB)
# - P100  (16 GB VRAM): "Qwen3.8-27B-UD-IQ3_S.gguf"  (~11.5 GB) or "Qwen3.8-27B-UD-IQ4_XS.gguf"
HF_REPO = "unsloth/Qwen3.8-27B-GGUF"
HF_FILENAME = "Qwen3.8-27B-UD-IQ4_XS.gguf"

# Optional: set if you attached the GGUF as a Kaggle Dataset/Model (/kaggle/input/...)
LOCAL_MODEL_PATH = ""

CONTEXT_LEN = 8192
PORT = 8000
BIN_DIR = "/tmp/llama-bin"

# Pick storage directory with the most free space (/tmp often has more room than /kaggle/working)
candidates = [p for p in ["/tmp", "/kaggle/working"] if os.path.exists(p)]
STORAGE_ROOT = max(candidates, key=lambda p: shutil.disk_usage(p).free)
MODELS_DIR = os.path.join(STORAGE_ROOT, "models")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(BIN_DIR, exist_ok=True)

for p in candidates:
    free_gb = shutil.disk_usage(p).free / (1024**3)
    print(f"Free disk at {p}: {free_gb:.1f} GB")
print("Selected MODELS_DIR:", MODELS_DIR)

gpu_check = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
HAS_GPU = gpu_check.returncode == 0
print("GPU detected:" if HAS_GPU else "No GPU detected (CPU mode)")
if HAS_GPU:
    print(gpu_check.stdout.strip())

## 2. Download Prebuilt `llama.cpp` (`llama-server`)

In [ ]:
import json, urllib.request, tarfile

# Fetch latest b* release from ggml-org/llama.cpp that already has finished CI assets
req = urllib.request.Request(
    "https://api.github.com/repos/ggml-org/llama.cpp/releases?per_page=20",
    headers={"User-Agent": "kaggle-notebook"}
)
with urllib.request.urlopen(req) as resp:
    releases = json.loads(resp.read().decode())

if HAS_GPU:
    def needed_for(tag):
        return [
            f"llama-{tag}-bin-ubuntu-cuda-12.8-x64.tar.gz",
            f"cudart-llama-{tag}-bin-ubuntu-cuda-12.8-x64.tar.gz",
        ]
else:
    def needed_for(tag):
        return [f"llama-{tag}-bin-ubuntu-x64.tar.gz"]

release = next(
    (
        r for r in releases
        if r.get("tag_name", "").startswith("b")
        and all(
            any(a["name"] == fn for a in r.get("assets", []))
            for fn in needed_for(r["tag_name"])
        )
    ),
    None,
)
if not release:
    raise RuntimeError("No recent llama.cpp release with matching prebuilt assets found.")

tag = release["tag_name"]
assets = {a["name"]: a["browser_download_url"] for a in release["assets"]}
print("Using llama.cpp release:", tag)

for fname in needed_for(tag):
    url = assets[fname]
    tgz_path = f"/tmp/{fname}"
    print(f"Downloading {fname}...")
    urllib.request.urlretrieve(url, tgz_path)
    with tarfile.open(tgz_path, "r:gz") as tf:
        for member in tf.getmembers():
            # Strip top-level directory so binaries & .so files land directly in BIN_DIR
            parts = member.name.split("/", 1)
            if len(parts) < 2 or not parts[1]:
                continue
            member.name = parts[1]
            tf.extract(member, BIN_DIR)
    os.remove(tgz_path)

LLAMA_SERVER = os.path.join(BIN_DIR, "llama-server")
os.chmod(LLAMA_SERVER, 0o755)
print("Ready:", LLAMA_SERVER)

## 3. Download `Qwen3.8-27B` GGUF Model

In [ ]:
!pip install -q huggingface_hub openai
from huggingface_hub import hf_hub_download

if LOCAL_MODEL_PATH and os.path.exists(LOCAL_MODEL_PATH):
    MODEL_PATH = LOCAL_MODEL_PATH
else:
    MODEL_PATH = hf_hub_download(
        repo_id=HF_REPO,
        filename=HF_FILENAME,
        local_dir=MODELS_DIR,
    )

print("Model ready at:", MODEL_PATH)

## 4. Start `llama-server` (Native OpenAI-Compatible Endpoint)
Uses `--jinja`, Flash Attention (`-fa on`), and `-ngl 99` to offload all layers across available GPUs (e.g. both T4s).

In [ ]:
import time

subprocess.run(["pkill", "-f", "llama-server"], check=False)
time.sleep(0.5)

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = f"{BIN_DIR}:{env.get('LD_LIBRARY_PATH', '')}"

cmd = [
    LLAMA_SERVER,
    "-m", MODEL_PATH,
    "--host", "0.0.0.0",
    "--port", str(PORT),
    "-c", str(CONTEXT_LEN),
    "-ngl", "99" if HAS_GPU else "0",
    "-fa", "on",
    "--jinja",
    "--alias", "qwen3.8-27b",
]

log_path = "/tmp/llama_server.log"
server_proc = subprocess.Popen(
    cmd,
    stdout=open(log_path, "w"),
    stderr=subprocess.STDOUT,
    env=env,
)

# Wait for model to finish loading into VRAM/RAM (can take 30-90s for 27B)
for i in range(180):
    if server_proc.poll() is not None:
        print(open(log_path).read())
        raise RuntimeError(f"llama-server exited early with code {server_proc.returncode}")
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health") as r:
            body = r.read().decode()
            if '"ok"' in body or r.status == 200:
                print(f"llama-server ready in {i}s:", body)
                break
    except Exception:
        time.sleep(1)
else:
    print(open(log_path).read())
    raise TimeoutError("Timed out waiting for llama-server /health")

## 5. Test Locally with the OpenAI Python SDK

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=f"http://127.0.0.1:{PORT}/v1", api_key="not-needed")

resp = client.chat.completions.create(
    model="qwen3.8-27b",
    messages=[{"role": "user", "content": "Explain how GGUF quantization works in 3 concise bullet points."}],
    max_tokens=256,
    temperature=0.6,
)
print(resp.choices[0].message.content)

## 6. Expose Publicly via Cloudflare Tunnel
Run this cell to get a public `https://*.trycloudflare.com` URL. Use `https://<your-url>.trycloudflare.com/v1` as `base_url` in any OpenAI client.

In [ ]:
%%bash
if [ ! -f /usr/local/bin/cloudflared ]; then
    wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    chmod +x /usr/local/bin/cloudflared
fi

cloudflared tunnel --url http://localhost:8000